In [ ]:
import pandas as pd
import os

RAW_DIR = "../data/01_raw"
INTER_DIR = "../data/02_intermediate"
os.makedirs(INTER_DIR, exist_ok=True)

print("Starte Schema Mapping für Bevölkerung...")

# ==========================================
# BEVÖLKERUNG: Schema Translation auf Zielschema
# ==========================================
df_bev = pd.read_csv(os.path.join(RAW_DIR, 'bevoelkerung_roh.csv'), sep=';')

# Jahr aus dem Stichtags-Format extrahieren (z.B. "1984-12-31" -> 1984)
df_bev['jahr'] = df_bev['time'].astype(str).str[:4].astype(int)

# Genesis liefert Striche "-" für fehlende Werte -> Numerisch konvertieren
df_bev['value'] = pd.to_numeric(df_bev['value'], errors='coerce').fillna(0)

# Nationalitäts-Bezeichner auf einheitliches Schema normieren
df_bev['nationalitaet'] = df_bev['2_variable_attribute_label'].replace({
    'Deutsche': 'Deutsch',
    'Ausländer': 'Nichtdeutsch',
    'Insgesamt': 'Gesamt'
})

# Bundesländer-Granularität auflösen: Aufsummieren auf nationale Ebene pro Jahr+Nationalität
df_bev = df_bev.groupby(['jahr', 'nationalitaet'])['value'].sum().reset_index()

df_bev = df_bev.rename(columns={'value': 'bevoelkerung_gesamt'})
df_bev['bevoelkerung_gesamt'] = df_bev['bevoelkerung_gesamt'].astype(int)

df_bev.to_csv(os.path.join(INTER_DIR, 'bevoelkerung_mapped.csv'), index=False, sep=';')
print(f"Bevölkerung gemappt (Zeilen: {len(df_bev)})")

display(df_bev.sample(5))

In [ ]:
# ==========================================
# PKS: Unpivot von Wide-Format auf Long-Format
# ==========================================
# Die PKS speichert die Nationalität als separate Spalten (anzahl_deutsch, anzahl_nichtdeutsch, ...)
# -> Für den späteren Join brauchen wir eine eigene Spalte "nationalitaet" (Long-Format via Melt)
print("Starte Schema Mapping für PKS...")

df_pks = pd.read_csv(os.path.join(INTER_DIR, 'pks_cleaned.csv'), sep=';')

nat_map_pks = {
    'anzahl_deutsch': 'Deutsch',
    'anzahl_nichtdeutsch': 'Nichtdeutsch',
    'anzahl_gesamt': 'Gesamt'
}

df_pks = df_pks.melt(
    id_vars=['Jahr', 'Deliktschlüssel', 'Straftat'],
    value_vars=['anzahl_deutsch', 'anzahl_nichtdeutsch', 'anzahl_gesamt'],
    var_name='nat_raw',
    value_name='tatverdaechtige'
)

df_pks['nationalitaet'] = df_pks['nat_raw'].map(nat_map_pks)
df_pks = df_pks.drop(columns=['nat_raw'])

# Spaltennamen auf einheitliches Schema angleichen
df_pks = df_pks.rename(columns={
    'Jahr': 'jahr',
    'Deliktschlüssel': 'straftat_code',
    'Straftat': 'straftat_bezeichnung'
})

df_pks = df_pks[['jahr', 'straftat_code', 'straftat_bezeichnung', 'nationalitaet', 'tatverdaechtige']]

df_pks.to_csv(os.path.join(INTER_DIR, 'pks_mapped.csv'), index=False, sep=';')
print(f"PKS gemappt (Zeilen: {len(df_pks)})")

display(df_pks.sample(5))